# First Title
## Second Title
- Item 1
- Item 2
- Item 3 
**abcde**

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
df=pd.read_csv("RFM_raw data.csv") 
df.head

<bound method NDFrame.head of           Date     InvoNo CustomerKey     Amount
0     20180402  AP1803827       A1067   836.0000
1     20180403  AP1808235       A1129   836.0000
2     20180404  AP1807305       A1133   836.0000
3     20180405  AP1806145       A0464   836.0000
4     20180405  AP1803092       A0383   836.0000
...        ...        ...         ...        ...
9376  20190102  JA1905084       A0922  2472.7020
9377  20190102  JA1908815       A0738  3500.7984
9378  20190104  JA1903532       A0632  2365.4400
9379  20190104  JA1902149       A1036  6020.8400
9380  20190104  JA1907784       A1057   205.7000

[9381 rows x 4 columns]>

In [10]:
# 1. 確保 Date 欄位為日期格式（若已經是 datetime 可省略此行）
df['Date'] = pd.to_datetime(df['Date'])

# 2. 取得整個資料集的「最大日期」作為基準日
max_date = df['Date'].max()

# 3. 進行 Group By 聚合計算
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),                           # 發票編號計數（建議用 nunique 計算不重複發票張數）
    M=('Amount', 'sum'),                               # 金額加總
    R=('Date', lambda x: (max_date - x.max()).days)    # 全局最大日期 - 該客戶最後交易日期的天數差
).reset_index()

# 4. 重新命名欄位並調整順序
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# 查看結果
print(result)

    customer   F            M  R
0      A0350  25  51238.11440  0
1      A0351   6  17890.31200  0
2      A0352  14  46424.03852  0
3      A0353  14  27607.68636  0
4      A0354   6   9973.92064  0
..       ...  ..          ... ..
727    A1413  23  52738.79456  0
728    A1415  20  53019.49664  0
729    A1416  24  49938.38378  0
730    A1419  22  64257.84520  0
731    A1420  14  25054.18342  0

[732 rows x 4 columns]


In [14]:
# 1. 確保轉為日期格式
df['Date'] = pd.to_datetime(df['Date'])

# 2. 全資料最大日期
max_date = df['Date'].max()

# 3. 分組計算：先抓出每位客戶的最後交易日 last_date
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'count'),
    M=('Amount', 'sum'),
    last_date=('Date', 'max')
).reset_index()

# 4. 向量化計算 R（全局最大日 - 客戶最後交易日）
result['R'] = (max_date - result['max_date']).dt.days

# 5. 整理欄位與順序
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

print(result)

KeyError: 'max_date'

In [15]:
df['Date'] = pd.to_datetime(df['Date'])

# 步驟 2：取出全資料庫的最大日期（注意是給變數 max_date，不是寫回 df['Date']）
max_date = df['Date'].max()
print(f"【全資料庫最大日期】: {max_date}")

# 步驟 3：先分組計算 F、M，以及每個客戶的「最後交易日 (last_date)」
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),
    M=('Amount', 'sum'),
    last_date=('Date', 'max')
).reset_index()

# 步驟 4：檢查 A350 的 last_date 是不是真的是 2018-12-26
a350_check = result[result['CustomerKey'] == 'A350']
print("【A350 分組後的資料】:")
print(a350_check[['CustomerKey', 'last_date']])

# 步驟 5：用向量化相減計算 R（不要用 lambda，最安全）
result['R'] = (max_date - result['last_date']).dt.days

# 步驟 6：調整欄位名稱與順序
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# 步驟 7：查看成果（特別印出前幾筆與 A350）
print("\n【最終結果前 5 筆】:")
print(result.head())

print("\n【A350 最終結果】:")
print(result[result['customer'] == 'A350'])

【全資料庫最大日期】: 1970-01-01 00:00:00.020190104
【A350 分組後的資料】:
Empty DataFrame
Columns: [CustomerKey, last_date]
Index: []

【最終結果前 5 筆】:
  customer   F            M  R
0    A0350  25  51238.11440  0
1    A0351   6  17890.31200  0
2    A0352  14  46424.03852  0
3    A0353  14  27607.68636  0
4    A0354   6   9973.92064  0

【A350 最終結果】:
Empty DataFrame
Columns: [customer, F, M, R]
Index: []


In [16]:
# 1. 確保 Date 欄位為 datetime 日期格式
df['Date'] = pd.to_datetime(df['Date'])

# 2. 取得「全資料集」的最大日期（儲存至獨立變數，切勿覆蓋 df['Date']）
max_date = df['Date'].max()

# 3. 依 CustomerKey 分組聚合
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),          # 發票計數（nunique 為不重複發票張數；若要算紀錄筆數可改 'count'）
    M=('Amount', 'sum'),              # 金額加總
    last_date=('Date', 'max')         # 該客戶最後一次交易日期
).reset_index()

# 4. 計算 R（全表最大日期 - 該客戶最後交易日）並轉為天數整數
result['R'] = (max_date - result['last_date']).dt.days

# 5. 欄位改名並依指定順序排列：customer, F, M, R
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# 查看結果
print(result)

    customer   F            M  R
0      A0350  25  51238.11440  0
1      A0351   6  17890.31200  0
2      A0352  14  46424.03852  0
3      A0353  14  27607.68636  0
4      A0354   6   9973.92064  0
..       ...  ..          ... ..
727    A1413  23  52738.79456  0
728    A1415  20  53019.49664  0
729    A1416  24  49938.38378  0
730    A1419  22  64257.84520  0
731    A1420  14  25054.18342  0

[732 rows x 4 columns]


In [18]:
# 1. 重新讀檔（請替換為你的檔名）
# df = pd.read_csv("RFM_raw data.csv")

# 2. 轉日期
df['Date'] = pd.to_datetime(df['Date'])

# 3. 取出全局基準日
ref_date = df['Date'].max()

# 4. 聚合計算
result = df.groupby('CustomerKey').agg(
    customer=('CustomerKey', 'first'),
    F=('InvoNo', 'nunique'),
    M=('Amount', 'sum'),
    R=('Date', lambda x: (ref_date - x.max()).days)
).reset_index(drop=True)

# 5. 指定欄位順序
result = result[['customer', 'F', 'M', 'R']]

# 印出 A350 確認
print(result[result['customer'] == 'A350'])
print(result)

Empty DataFrame
Columns: [customer, F, M, R]
Index: []
    customer   F            M  R
0      A0350  25  51238.11440  0
1      A0351   6  17890.31200  0
2      A0352  14  46424.03852  0
3      A0353  14  27607.68636  0
4      A0354   6   9973.92064  0
..       ...  ..          ... ..
727    A1413  23  52738.79456  0
728    A1415  20  53019.49664  0
729    A1416  24  49938.38378  0
730    A1419  22  64257.84520  0
731    A1420  14  25054.18342  0

[732 rows x 4 columns]


In [19]:
df['CustomerKey'] = df['CustomerKey'].astype(str).str.strip()
df['Date'] = pd.to_datetime(df['Date'])

# 1. 抓出全表最大日
t_max = df['Date'].max()

# 2. 抓出 A350 的所有紀錄與最大日
a350_df = df[df['CustomerKey'] == 'A350']
a350_max = a350_df['Date'].max()

print("【診斷 1】全表最大日期 (t_max) :", t_max)
print("【診斷 2】A350 最大日期 (a350_max):", a350_max)
print("【診斷 3】兩者直接相減天數        :", (t_max - a350_max).days)
print("\n【診斷 4】A350 的全部交易明細:")
print(a350_df[['Date', 'InvoNo', 'Amount']])

【診斷 1】全表最大日期 (t_max) : 1970-01-01 00:00:00.020190104
【診斷 2】A350 最大日期 (a350_max): NaT
【診斷 3】兩者直接相減天數        : nan

【診斷 4】A350 的全部交易明細:
Empty DataFrame
Columns: [Date, InvoNo, Amount]
Index: []


In [20]:


# -------------------------------------------------------------
# 1. 修正日期格式問題（依據你資料原本的長相選擇，通常是純數字 20190104）
# -------------------------------------------------------------
# 先把日期轉成乾淨的字串格式（去除小數點如 20190104.0）
clean_date_str = df['Date'].astype(str).str.split('.').str[0].str.strip()

# 使用指定的格式解析（如果是 2019/1/4 或 20190104 都能安全解析）
df['Date'] = pd.to_datetime(clean_date_str, format='mixed')

# -------------------------------------------------------------
# 2. 處理客戶代號格式（全部轉大寫、去空白）
# -------------------------------------------------------------
df['CustomerKey'] = df['CustomerKey'].astype(str).str.strip().str.upper()

# -------------------------------------------------------------
# 3. 計算基準日與進行 RFM 轉換
# -------------------------------------------------------------
max_date = df['Date'].max()
print("【修正後的全表最大日期】:", max_date)

# 分組聚合
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),
    M=('Amount', 'sum'),
    last_date=('Date', 'max')
).reset_index()

# 計算 R（天數差）
result['R'] = (max_date - result['last_date']).dt.days

# 整理欄位
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# -------------------------------------------------------------
# 4. 驗證結果
# -------------------------------------------------------------
# 模糊搜尋包含 350 的客戶，看看它到底叫什麼名字
print("\n【尋找包含 350 的客戶結果】:")
print(result[result['customer'].str.contains('350', na=False)])

【修正後的全表最大日期】: 1970-01-01 00:00:00

【尋找包含 350 的客戶結果】:
    customer   F            M  R
0      A0350  25  51238.11440  0
686    A1350   6  16114.82872  0


In [22]:
import pandas as pd

# -------------------------------------------------------------
# 步驟 1：重新讀取原始檔案（請換成你的真實路徑或檔案名稱）
# -------------------------------------------------------------

df = pd.read_csv("RFM_raw data.csv")       # 如果是 csv

# -------------------------------------------------------------
# 步驟 2：正確解析 YYYYMMDD 純數字日期
# -------------------------------------------------------------
# 關鍵：先轉成純整數字串（去掉可能的小數點 .0），再用 %Y%m%d 解析
df['Date'] = df['Date'].astype(str).str.split('.').str[0].str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

# 檢查一下全表最大日期，此時「必須」顯示 2019-01-04
max_date = df['Date'].max()
print("【正確解析後的全表最大日期】:", max_date)

# -------------------------------------------------------------
# 步驟 3：清理客戶編號格式（去掉空白）
# -------------------------------------------------------------
df['CustomerKey'] = df['CustomerKey'].astype(str).str.strip()

# -------------------------------------------------------------
# 步驟 4：分組計算 RFM
# -------------------------------------------------------------
result = df.groupby('CustomerKey').agg(
    F=('InvoNo', 'nunique'),
    M=('Amount', 'sum'),
    last_date=('Date', 'max')
).reset_index()

# 計算 R（天數差）
result['R'] = (max_date - result['last_date']).dt.days

# 欄位重新命名與排序
result = result.rename(columns={'CustomerKey': 'customer'})
result = result[['customer', 'F', 'M', 'R']]

# -------------------------------------------------------------
# 步驟 5：驗證 A0350 的結果
# -------------------------------------------------------------
print("\n【A0350 的正確結果】:")
print(result)

【正確解析後的全表最大日期】: 2019-01-04 00:00:00

【A0350 的正確結果】:
    customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]
